In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install "spacy<3.3.0"

In [ ]:
!pip install "thinc==8.1.10"

In [ ]:
!pip install numpy==1.26.4 datasets==2.18.0

In [ ]:
!pip install transformers==4.41.1 peft==0.9.0 accelerate==0.29.1

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Load and prepare data
with open('/content/drive/MyDrive/train_pos_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_pos_content = file.readlines()

with open('/content/drive/MyDrive/train_neg_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_neg_content = file.readlines()

train_pos = pd.DataFrame(train_pos_content, columns=['tweet'])
train_pos['label'] = 1
train_neg = pd.DataFrame(train_neg_content, columns=['tweet'])
train_neg['label'] = 0
train = pd.concat([train_pos, train_neg], ignore_index=True)

# Split data
train_df, val_df = train_test_split(train, test_size=0.1, random_state=42)

# Load tokenizer
model_id = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenize
def tokenize(example):
    return tokenizer(example["tweet"], padding="max_length", truncation=True, max_length=40)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2, ignore_mismatched_sizes=True)
print(train_dataset)
print(train_dataset["input_ids"])
print(val_dataset)
print(model)

In [ ]:
print(train_dataset["attention_mask"])

In [ ]:
from sklearn.metrics import accuracy_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [ ]:
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

In [ ]:
# Parameters to tune:
# rank
# lora_alpha
# target_modules


# Apply LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=4,
    bias="none",
    lora_alpha=4,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
for name, param in model.named_parameters():
    if "classifier" in name and param.requires_grad:
        print(f"{name} is trainable")

# Training args
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Twitter_RoBERTa_LoRA",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=5e-4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    warmup_ratio=0.03,          # 6% warmup
    lr_scheduler_type="linear", # linear decay after warmup
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train
model.config.use_cache = False

from numpy.core.multiarray import _reconstruct
torch.serialization.add_safe_globals([_reconstruct])

trainer.train()

In [ ]:
from datetime import datetime

# Load and preprocess test set
with open('/content/drive/MyDrive/test_cleaned.txt', 'r', encoding='utf-8') as file:
    test_content = file.readlines()

test_df = pd.DataFrame(test_content, columns=["tweet"])
test_dataset = Dataset.from_pandas(test_df)

def tokenize(example):
    return tokenizer(example["tweet"], padding="max_length", truncation=True, max_length=40)

test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Run prediction
predictions = trainer.predict(test_dataset)
logits = predictions.predictions  # shape: (N, 2)

# Apply softmax to get probabilities
probs = torch.softmax(torch.tensor(logits), dim=-1)

# Get predicted class (0 or 1)
y_pred = torch.argmax(probs, dim=1).numpy()

# Optional: Convert class 0 to -1 (if needed for your evaluation or submission format)
y_pred[y_pred == 0] = -1


# Prepare submission
df = pd.read_csv('/content/drive/MyDrive/sample_submission.csv')
if len(y_pred) != len(df):
    print('Wrong size')
else:
    df["Prediction"] = y_pred
    time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    filename = "submission_" + time + ".csv"
    store_path = '/content/drive/MyDrive/Submissions/' + filename
    df.to_csv(store_path, index=False)
    print("Submission stored at:", store_path)
